In [8]:
import os
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

ROOT = "https://odre.opendatasoft.com/api/explore/v2.1/catalog/datasets"
URL_CONS = f"{ROOT}/eco2mix-national-cons-def/records"
URL_TR = f"{ROOT}/eco2mix-national-tr/records"

FICHIER = "eco2mix_15min.csv"
DEBUT = pd.Timestamp("2012-01-01", tz="Europe/Paris")  # 31/12/2011 23:00 UTC = premier jour du consolidé
FIN_CONS = pd.Timestamp("2026-09-01", tz="Europe/Paris")  # premier jour absent du consolidé
COLONNES_TEXTE = ["perimetre", "nature", "date", "heure"]
WORKERS = 20  # réduire si l'API renvoie des erreurs 429


def url_pour_mois(mois):
    return URL_CONS if mois < FIN_CONS else URL_TR


def telecharger_mois(mois):
    """Télécharge un mois complet, pagination incluse. Renvoie (mois, DataFrame) ou (mois, None)."""
    fin = mois + pd.DateOffset(months=1)
    where = (
        f"date_heure >= '{mois.tz_convert('UTC').isoformat()}' "
        f"AND date_heure < '{fin.tz_convert('UTC').isoformat()}'"
    )
    url = url_pour_mois(mois)
    lignes = []
    offset = 0
    try:
        while True:
            params = {"where": where, "order_by": "date_heure", "limit": 100, "offset": offset}
            r = requests.get(url, params=params, timeout=30)
            r.raise_for_status()
            data = r.json()
            lignes.extend(data["results"])
            offset += 100
            if offset >= data["total_count"] or not data["results"]:
                break
    except Exception as e:
        print(f"ERREUR {mois:%Y-%m} : {e}")
        return mois, None
    return mois, pd.DataFrame(lignes)


def charger_connus():
    """Renvoie les horodatages déjà présents dans le CSV."""
    if not os.path.exists(FICHIER):
        return pd.DatetimeIndex([], tz="UTC")
    idx = pd.read_csv(FICHIER, usecols=["date_heure"])["date_heure"]
    return pd.DatetimeIndex(pd.to_datetime(idx, utc=True))


def mois_manquants(connus):
    """Mois (heure de Paris) contenant au moins un quart d'heure absent du CSV."""
    maintenant = pd.Timestamp.now(tz="UTC").floor("15min")
    grille = pd.date_range(DEBUT.tz_convert("UTC"), maintenant, freq="15min")
    manquants = grille.difference(connus).tz_convert("Europe/Paris")
    mois = {pd.Timestamp(year=t.year, month=t.month, day=1, tz="Europe/Paris") for t in manquants}
    return sorted(mois)


def ajouter_au_csv(df_mois, connus):
    """Ajoute au CSV les lignes dont l'horodatage n'existe pas encore. Renvoie l'index connu mis à jour."""
    df_mois = df_mois.set_index("date_heure")
    df_mois.index = pd.to_datetime(df_mois.index, utc=True)
    df_mois = df_mois[~df_mois.index.isin(connus)].copy()
    if df_mois.empty:
        return connus

    df_mois = df_mois[~df_mois.index.duplicated(keep="last")]
    for col in df_mois.columns:
        if col not in COLONNES_TEXTE:
            df_mois[col] = pd.to_numeric(df_mois[col], errors="coerce")
    df_mois.index.name = "date_heure"

    if not os.path.exists(FICHIER):
        df_mois.to_csv(FICHIER)
        return connus.append(df_mois.index)

    colonnes_csv = list(pd.read_csv(FICHIER, nrows=0, index_col="date_heure").columns)
    nouvelles = [c for c in df_mois.columns if c not in colonnes_csv]

    if nouvelles:
        # Le temps réel ajoute des colonnes absentes du consolidé : on réécrit le fichier une fois
        complet = pd.read_csv(FICHIER, index_col="date_heure")
        complet.index = pd.to_datetime(complet.index, utc=True)
        complet = pd.concat([complet, df_mois])
        complet.index.name = "date_heure"
        complet.sort_index().to_csv(FICHIER)
        return pd.DatetimeIndex(complet.index)

    df_mois.reindex(columns=colonnes_csv).to_csv(FICHIER, mode="a", header=False)
    return connus.append(df_mois.index)


def mettre_a_jour():
    connus = charger_connus()
    print("Horodatages déjà présents :", len(connus))

    a_faire = mois_manquants(connus)
    print("Mois à télécharger :", len(a_faire))

    if not a_faire:
        print("Rien de nouveau à télécharger")
        return

    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        # map renvoie les résultats dans l'ordre des mois, donc le CSV reste trié
        for mois, df_mois in pool.map(telecharger_mois, a_faire):
            if df_mois is None:
                continue
            if df_mois.empty:
                print(f"{mois:%Y-%m} : aucune donnée")
                continue
            connus = ajouter_au_csv(df_mois, connus)
            print(f"{mois:%Y-%m} : {len(df_mois)} lignes reçues, CSV mis à jour")


mettre_a_jour()

df = pd.read_csv(FICHIER, index_col="date_heure")
df.index = pd.to_datetime(df.index, utc=True)
df = df.sort_index()

print(df.shape)
print(df.index.min(), df.index.max())
print(df.index.to_series().diff().value_counts())

KeyboardInterrupt: 

In [11]:
import os
import time
import requests
import pandas as pd

URL_PRIX = "https://api.energy-charts.info/price"
FICHIER_PRIX = "prix_day_ahead_fr.csv"
FICHIER_DEBUT = "debut_historique.txt"
BORNE_MIN = pd.Timestamp("2015-01-01", tz="Europe/Paris")
PAUSE = 2.0          # secondes entre deux requêtes
MAX_TENTATIVES = 5   # essais en cas de 429


class ApiIndisponible(Exception):
    pass


def get_json(params):
    """Une requête, avec pause avant chaque appel. 404 = pas de donnée, 429 = attente et reprise."""
    attente = 30
    for tentative in range(1, MAX_TENTATIVES + 1):
        time.sleep(PAUSE)
        r = requests.get(URL_PRIX, params=params, timeout=60)
        if r.status_code == 200:
            return r.json()
        if r.status_code == 404:
            return {}  # pas de donnée pour cette période
        if r.status_code == 429:
            retry = r.headers.get("Retry-After", "")
            secondes = float(retry) if retry.isdigit() else attente
            print(f"  429 reçu, pause {secondes:.0f}s (essai {tentative}/{MAX_TENTATIVES})")
            time.sleep(secondes)
            attente = min(attente * 2, 900)
            continue
        r.raise_for_status()  # autres erreurs : affichées, pas masquées
    raise ApiIndisponible("toujours en 429, réessaie plus tard")


def premier_du_mois(ts):
    return pd.Timestamp(year=ts.year, month=ts.month, day=1, tz="Europe/Paris")


def a_des_donnees(jour):
    debut = jour.tz_convert("UTC").isoformat()
    fin = (jour + pd.Timedelta(days=1)).tz_convert("UTC").isoformat()
    d = get_json({"bzn": "FR", "start": debut, "end": fin})
    return len(d.get("unix_seconds", [])) > 0


def trouver_debut(borne_min, borne_max):
    """Premier jour avec des prix, par dichotomie."""
    if a_des_donnees(borne_min):
        print("Des données existent dès BORNE_MIN : le début réel est peut-être plus ancien")
        return borne_min
    if not a_des_donnees(borne_max):
        raise ValueError("Aucune donnée jusqu'à borne_max : vérifier l'API")

    lo, hi = borne_min, borne_max  # lo : pas de données, hi : données
    while (hi - lo).days > 1:
        milieu = (lo + (hi - lo) // 2).normalize()
        if a_des_donnees(milieu):
            hi = milieu
        else:
            lo = milieu
    return hi


def debut_historique():
    """Date de début, lue dans le cache si elle existe, sinon calculée une seule fois."""
    if os.path.exists(FICHIER_DEBUT):
        with open(FICHIER_DEBUT) as f:
            debut = pd.Timestamp(f.read().strip()).tz_convert("Europe/Paris")
        print("Début de l'historique (cache) :", debut.date())
        return debut
    debut = trouver_debut(BORNE_MIN, pd.Timestamp.now(tz="Europe/Paris").normalize())
    with open(FICHIER_DEBUT, "w") as f:
        f.write(debut.isoformat())
    print("Début de l'historique :", debut.date())
    return debut


def csv_present():
    return os.path.exists(FICHIER_PRIX) and os.path.getsize(FICHIER_PRIX) > 0


def charger_connus():
    if not csv_present():
        return pd.DatetimeIndex([], tz="UTC")
    dates = pd.read_csv(FICHIER_PRIX, usecols=["dates"])["dates"]
    return pd.DatetimeIndex(pd.to_datetime(dates, utc=True))


def telecharger_mois(mois):
    fin = mois + pd.DateOffset(months=1)
    d = get_json({
        "bzn": "FR",
        "start": mois.tz_convert("UTC").isoformat(),
        "end": fin.tz_convert("UTC").isoformat(),
    })
    if not d.get("unix_seconds"):
        return pd.DataFrame()
    df = pd.DataFrame({
        "dates": pd.to_datetime(d["unix_seconds"], unit="s", utc=True),
        "price": pd.to_numeric(d["price"], errors="coerce"),
    })
    return df.drop_duplicates("dates").set_index("dates").sort_index()


def mois_a_telecharger(connus, debut):
    """Mois avec au moins un jour absent du CSV, plus les deux derniers mois (toujours revus)."""
    aujourdhui = pd.Timestamp.now(tz="Europe/Paris").normalize()
    mois_courant = premier_du_mois(aujourdhui)
    jours_connus = set(connus.tz_convert("Europe/Paris").date) if len(connus) else set()

    a_faire = []
    for m in pd.date_range(premier_du_mois(debut), mois_courant, freq="MS", tz="Europe/Paris"):
        if m >= mois_courant - pd.DateOffset(months=1):
            a_faire.append(m)
            continue
        fin_mois = min(m + pd.DateOffset(months=1), aujourdhui + pd.Timedelta(days=1))
        jours = [d.date() for d in pd.date_range(m, fin_mois, freq="D", inclusive="left")]
        if any(j not in jours_connus for j in jours):
            a_faire.append(m)
    return a_faire


def ajouter_au_csv(nouvelles):
    nouvelles = nouvelles.copy()
    nouvelles.index.name = "dates"
    nouvelles.to_csv(FICHIER_PRIX, mode="a", header=not csv_present())


def trier_csv():
    if not csv_present():
        return
    df = pd.read_csv(FICHIER_PRIX, index_col="dates")
    df.index = pd.to_datetime(df.index, utc=True)
    df = df[~df.index.duplicated(keep="last")].sort_index()
    df.index.name = "dates"
    df.to_csv(FICHIER_PRIX)


def mettre_a_jour_prix():
    try:
        debut = debut_historique()
    except ApiIndisponible as e:
        print(f"Arrêt : {e}")
        return

    connus = charger_connus()
    print("Déjà dans le CSV :", len(connus), "prix")

    a_faire = mois_a_telecharger(connus, debut)
    print("Mois à télécharger :", len(a_faire))

    for mois in a_faire:  # une requête après l'autre
        try:
            df_mois = telecharger_mois(mois)
        except ApiIndisponible as e:
            print(f"Arrêt à {mois:%Y-%m} : {e}. La progression est sauvegardée.")
            break
        if df_mois.empty:
            print(f"{mois:%Y-%m} : aucune donnée")
            continue
        nouvelles = df_mois[~df_mois.index.isin(connus)]
        if nouvelles.empty:
            print(f"{mois:%Y-%m} : déjà complet")
            continue
        ajouter_au_csv(nouvelles)  # sauvegarde à chaque mois
        connus = connus.append(nouvelles.index)
        print(f"{mois:%Y-%m} : {len(nouvelles)} prix ajoutés")

    trier_csv()


mettre_a_jour_prix()

df_price = pd.read_csv(FICHIER_PRIX, index_col="dates")
df_price.index = pd.to_datetime(df_price.index, utc=True)
df_price = df_price.sort_index()
print(df_price.shape)
print(df_price.index.min(), df_price.index.max())
print(df_price.index.to_series().diff().value_counts())

  429 reçu, pause 24s (essai 1/5)
  429 reçu, pause 11s (essai 1/5)
  429 reçu, pause 25s (essai 1/5)
  429 reçu, pause 21s (essai 1/5)
  429 reçu, pause 25s (essai 1/5)
  429 reçu, pause 24s (essai 1/5)
Début de l'historique : 2015-01-04
Déjà dans le CSV : 0 prix
Mois à télécharger : 142
2015-01 : 744 prix ajoutés
  429 reçu, pause 16s (essai 1/5)
  429 reçu, pause 2s (essai 2/5)
2015-02 : 672 prix ajoutés
2015-03 : 743 prix ajoutés
  429 reçu, pause 17s (essai 1/5)
2015-04 : 720 prix ajoutés
  429 reçu, pause 18s (essai 1/5)
2015-05 : 744 prix ajoutés
2015-06 : 720 prix ajoutés
2015-07 : 744 prix ajoutés
  429 reçu, pause 23s (essai 1/5)
2015-08 : 744 prix ajoutés
  429 reçu, pause 17s (essai 1/5)
2015-09 : 720 prix ajoutés
2015-10 : 745 prix ajoutés
2015-11 : 720 prix ajoutés
  429 reçu, pause 7s (essai 1/5)
  429 reçu, pause 3s (essai 2/5)
2015-12 : 744 prix ajoutés
  429 reçu, pause 24s (essai 1/5)
2016-01 : 744 prix ajoutés
2016-02 : 696 prix ajoutés
2016-03 : 743 prix ajoutés
  

In [ ]:
jour = pd.Timestamp("2025-06-15", tz="Europe/Paris")
params = {
    "bzn": "FR",
    "start": jour.tz_convert("UTC").isoformat(),
    "end": (jour + pd.Timedelta(days=1)).tz_convert("UTC").isoformat(),
}
r = requests.get(URL_PRIX, params=params, timeout=30)
print(r.status_code)
print(r.text[:300])

200
{"license_info":"CC BY 4.0 (creativecommons.org/licenses/by/4.0) from Bundesnetzagentur | SMARD.de","unix_seconds":[1749938400,1749942000,1749945600,1749949200,1749952800,1749956400,1749960000,1749963600,1749967200,1749970800,1749974400,1749978000,1749981600,1749985200,1749988800,1749992400,17499960


In [12]:
import os
import time
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

URL_METEO = "https://historical-forecast-api.open-meteo.com/v1/forecast"
FICHIER_METEO = "meteo_horaire_paris.csv"
LAT, LON = 48.85, 2.35
VARIABLES = "temperature_2m,wind_speed_100m,shortwave_radiation,cloud_cover"
DEBUT = pd.Timestamp("2022-01-01")  # à vérifier : première date acceptée par l'API
FIN = pd.Timestamp.now().normalize() - pd.Timedelta(days=1)  # hier
WORKERS = 2
PAUSE = 1.0


def get_json(params):
    """Une requête avec pause. 429 = attente puis reprise. 400 = affiché et ignoré."""
    attente = 30
    for tentative in range(1, 6):
        time.sleep(PAUSE)
        r = requests.get(URL_METEO, params=params, timeout=60)
        if r.status_code == 200:
            return r.json()
        if r.status_code == 429:
            print(f"  429, pause {attente}s (essai {tentative}/5)")
            time.sleep(attente)
            attente = min(attente * 2, 900)
            continue
        if r.status_code == 400:
            print("  400 :", r.text[:200])
            return None
        r.raise_for_status()
    raise RuntimeError("429 persistant, réessayer plus tard")


def fetch_mois(mois):
    """Télécharge les heures d'un mois, bornées à [DEBUT, FIN]. Renvoie un DataFrame indexé par time."""
    debut = max(mois, DEBUT)
    fin = min(mois + pd.DateOffset(months=1) - pd.Timedelta(days=1), FIN)
    if debut > fin:
        return mois, pd.DataFrame()

    params = {
        "latitude": LAT,
        "longitude": LON,
        "start_date": debut.strftime("%Y-%m-%d"),
        "end_date": fin.strftime("%Y-%m-%d"),
        "hourly": VARIABLES,
        "timezone": "UTC",
    }
    d = get_json(params)
    if not d or "hourly" not in d:
        return mois, pd.DataFrame()

    df = pd.DataFrame(d["hourly"])
    df["time"] = pd.to_datetime(df["time"], utc=True)
    return mois, df.set_index("time").sort_index()


def charger_csv():
    if not os.path.exists(FICHIER_METEO) or os.path.getsize(FICHIER_METEO) == 0:
        return pd.DataFrame()
    df = pd.read_csv(FICHIER_METEO, index_col="time")
    df.index = pd.to_datetime(df.index, utc=True)
    return df.sort_index()


def mois_a_telecharger(connus):
    """Mois avec au moins une heure absente, plus les deux derniers mois (toujours revus)."""
    tous = pd.date_range(DEBUT.replace(day=1), FIN, freq="MS")
    revus = set(tous[-2:])
    a_faire = []
    for m in tous:
        heures = pd.date_range(max(m, DEBUT), min(m + pd.DateOffset(months=1), FIN + pd.Timedelta(days=1)),
                               freq="h", inclusive="left", tz="UTC")
        manquantes = heures.difference(connus) if len(connus) else heures
        if len(manquantes) > 0 or m in revus:
            a_faire.append(m)
    return a_faire


def mettre_a_jour_meteo():
    connus_df = charger_csv()
    connus = connus_df.index if len(connus_df) else pd.DatetimeIndex([], tz="UTC")
    print("Heures déjà présentes :", len(connus))

    a_faire = mois_a_telecharger(connus)
    print("Mois à télécharger :", len(a_faire))

    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        futures = [pool.submit(fetch_mois, m) for m in a_faire]
        for fut in as_completed(futures):
            mois, df_mois = fut.result()
            if df_mois.empty:
                print(f"{mois:%Y-%m} : aucune donnée")
                continue
            df_mois.index.name = "time"
            existe = os.path.exists(FICHIER_METEO) and os.path.getsize(FICHIER_METEO) > 0
            df_mois.to_csv(FICHIER_METEO, mode="a", header=not existe)  # sauvegarde mois par mois
            print(f"{mois:%Y-%m} : {len(df_mois)} heures écrites")

    # Nettoyage final : doublons (version la plus récente gardée), tri chronologique
    df = charger_csv()
    if len(df):
        df = df[~df.index.duplicated(keep="last")].sort_index()
        df.index.name = "time"
        df.to_csv(FICHIER_METEO)


mettre_a_jour_meteo()

df_meteo = charger_csv()
df_meteo_15 = (
    df_meteo
    .resample("15min")
    .asfreq()
    .interpolate(method="time")
)
print(df_meteo.shape)
print(df_meteo.index.min(), df_meteo.index.max())
print(df_meteo.isna().sum())

Heures déjà présentes : 0
Mois à télécharger : 58
2022-02 : 672 heures écrites
2022-01 : 744 heures écrites
2022-04 : 720 heures écrites
2022-03 : 744 heures écrites
2022-05 : 744 heures écrites
2022-06 : 720 heures écrites
2022-07 : 744 heures écrites
2022-08 : 744 heures écrites
2022-09 : 720 heures écrites
2022-10 : 744 heures écrites
2022-11 : 720 heures écrites
2022-12 : 744 heures écrites
2023-01 : 744 heures écrites
2023-02 : 672 heures écrites
2023-03 : 744 heures écrites
2023-04 : 720 heures écrites
2023-05 : 744 heures écrites
2023-06 : 720 heures écrites
2023-07 : 744 heures écrites
2023-08 : 744 heures écrites
2023-09 : 720 heures écrites
2023-10 : 744 heures écrites
2023-11 : 720 heures écrites
2023-12 : 744 heures écrites
2024-01 : 744 heures écrites
2024-02 : 696 heures écrites
2024-03 : 744 heures écrites
2024-04 : 720 heures écrites
2024-05 : 744 heures écrites
2024-06 : 720 heures écrites
2024-07 : 744 heures écrites
2024-08 : 744 heures écrites
2024-09 : 720 heures é